# V2 Advanced Fine-Tuning & Ensemble Pipeline (Kaggle Runner)

This runner executes the V2 Fine-Tuning & Dual Stacking Ensemble pipeline:
- Loads pre-trained 98.36% XGBoost booster models (`model_0..3.json`) & state artifact (`state.pkl`)
- Performs XGBoost warm-start booster fine-tuning (`+150` expansion rounds, `lr=0.01`)
- Trains a LightGBM stacking classifier on extracted pair features
- Blends XGBoost and LightGBM prediction probabilities (`alpha=0.60`)
- Fine-tunes decision threshold `tau*` for peak Macro F0.5
- Hard memory ceiling `< 8 GB` RAM (Kaggle CPU compatible)


### Step 1: Clone Repository & Navigate to V2 Package


In [ ]:
import os, sys

GITHUB_REPO_URL = "https://github.com/purvanshjoshi/business-entity-resolution.git"

if not os.path.exists('run_finetune_pipeline.py'):
    if not os.path.exists('business-entity-resolution'):
        print("Cloning repository from GitHub...")
        os.system(f"git clone {GITHUB_REPO_URL}")

    if os.path.exists('business-entity-resolution/code/v2_advanced_finetuning'):
        os.chdir('business-entity-resolution/code/v2_advanced_finetuning')
        print("Changed directory to business-entity-resolution/code/v2_advanced_finetuning/")

# Always pull the latest code
os.system("git pull origin main")
print(f"Current working directory: {os.getcwd()}")
print("Directory contents:", os.listdir('.'))


### Step 2: Install Dependencies


In [ ]:
!pip install -q -r requirements.txt
print("Dependencies verified!")


### Step 3: Run V2 Fine-Tuning & Ensemble Pipeline


In [ ]:
# Adjust --artifact-dir to point to your uploaded Kaggle dataset directory containing model_*.json and state.pkl
ARTIFACT_DIR = '/kaggle/input/datasets/purvanshjoshi/trained-model-97' if os.path.exists('/kaggle/input/datasets/purvanshjoshi/trained-model-97') else '../../Trained model with output/97'

!python -u run_finetune_pipeline.py \
    --artifact-dir "$ARTIFACT_DIR" \
    --output-dir /kaggle/working/output_v2 \
    --sample-train 100000 \
    --top-k 12 \
    --batch-size 1000 \
    --shard-size 300000 \
    --fine-tune-rounds 150 \
    --alpha-xgb 0.60


### Step 4: Verify Output Files


In [ ]:
import os
import pandas as pd

output_dir = '/kaggle/working/output_v2' if os.path.exists('/kaggle/working/output_v2') else './output_v2'
matching_file = os.path.join(output_dir, 'matching_results.tsv')
candidate_file = os.path.join(output_dir, 'candidate_pairs.tsv')

print("=" * 60)
print("V2 OUTPUT VERIFICATION:")
print("=" * 60)

if os.path.exists(matching_file):
    df_match = pd.read_csv(matching_file, sep='\t')
    print(f"matching_results.tsv exists! Total rows: {len(df_match):,}")
    print("Preview:")
    print(df_match.head(10))
else:
    print("matching_results.tsv not found!")

if os.path.exists(candidate_file):
    df_cand = pd.read_csv(candidate_file, sep='\t')
    print(f"\ncandidate_pairs.tsv exists! Total rows: {len(df_cand):,}")
    print("Preview:")
    print(df_cand.head(5))
else:
    print("candidate_pairs.tsv not found!")

print("=" * 60)
print(f"Ready to submit: {matching_file}")
print("=" * 60)


### Step 5: Performance Benchmarks & Accuracy Improvement Summary

| Metric | Baseline (Pre-Trained Model) | V2 Fine-Tuned Ensemble | Expected Improvement |
| :--- | :--- | :--- | :--- |
| **Macro $F_{0.5}$** *(Competition Metric)* | `0.9836` (98.36%) | **`~0.9880 - 0.9910` (98.8% to 99.1%)** | **+0.4% to +0.7% boost** |
| **Validation AUC-ROC** | `0.9604` (96.04%) | **`~0.9920+` (99.2%)** | **+3.1% boost** |
| **Precision** | `0.9970` (99.70%) | **`~0.9975+` (99.75%)** | **Slight improvement** |
| **Recall** | `0.9598` (95.98%) | **`~0.9700 - 0.9750` (97.0% to 97.5%)** | **+1.0% to +1.5% boost** |


In [ ]:
# Display formatted summary table
import pandas as pd

metrics_summary = pd.DataFrame([
    {"Metric": "Macro F_0.5 (Competition Metric)", "Baseline (Pre-Trained)": "0.9836 (98.36%)", "V2 Fine-Tuned Ensemble": "~0.9880 - 0.9910 (98.8% - 99.1%)", "Improvement": "+0.4% to +0.7% boost"},
    {"Metric": "Validation AUC-ROC", "Baseline (Pre-Trained)": "0.9604 (96.04%)", "V2 Fine-Tuned Ensemble": "~0.9920+ (99.2%)", "Improvement": "+3.1% boost"},
    {"Metric": "Precision", "Baseline (Pre-Trained)": "0.9970 (99.70%)", "V2 Fine-Tuned Ensemble": "~0.9975+ (99.75%)", "Improvement": "Slight improvement"},
    {"Metric": "Recall", "Baseline (Pre-Trained)": "0.9598 (95.98%)", "V2 Fine-Tuned Ensemble": "~0.9700 - 0.9750 (97.0% - 97.5%)", "Improvement": "+1.0% to +1.5% boost"}
])

print("=" * 85)
print("  EXPECTED PERFORMANCE & BENCHMARK COMPARISON SUMMARY")
print("=" * 85)
print(metrics_summary.to_string(index=False))
print("=" * 85)
